# SQL Fundamentals: Joins, CTEs & Window Functions

**Module:** 1 — Foundations & Data Engineering  
**Week:** 2, Day 1  

## Learning Objectives
- Create tables and insert data using SQLite
- Write JOINs: INNER, LEFT, RIGHT (emulated), CROSS
- Use GROUP BY with HAVING for filtered aggregations
- Write CTEs (Common Table Expressions) for readable queries
- Use window functions: ROW_NUMBER, RANK, LAG, running totals
- Combine SQL with Pandas via `sqlite3` + `pd.read_sql()`

## Resources
- [SQLBolt](https://sqlbolt.com/) — Interactive exercises
- [Mode Analytics SQL Tutorial](https://mode.com/sql-tutorial/)
- [SQLite Documentation](https://www.sqlite.org/lang.html)
- [Window Functions Explained](https://mode.com/sql-tutorial/sql-window-functions/)

In [ ]:
import sqlite3
import pandas as pd
import numpy as np

# Helper to run SQL and return a DataFrame
def sql(query: str, conn: sqlite3.Connection) -> pd.DataFrame:
    """Execute SQL and return results as a DataFrame."""
    return pd.read_sql(query, conn)

# Helper to run SQL without returning results
def execute(query: str, conn: sqlite3.Connection):
    conn.executescript(query)
    conn.commit()

---
## 1. Setting Up Our Database

We'll create an in-memory SQLite database with a realistic e-commerce schema.

In [ ]:
conn = sqlite3.connect(":memory:")

execute("""
-- Customers table
CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    email TEXT UNIQUE,
    tier TEXT CHECK(tier IN ('Gold', 'Silver', 'Bronze')),
    signup_date DATE,
    region TEXT
);

-- Products table
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name TEXT NOT NULL,
    category TEXT,
    price REAL,
    cost REAL
);

-- Orders table
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    order_date DATE,
    status TEXT CHECK(status IN ('completed', 'pending', 'cancelled', 'returned')),
    FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);

-- Order items (line items)
CREATE TABLE order_items (
    item_id INTEGER PRIMARY KEY AUTOINCREMENT,
    order_id INTEGER,
    product_id INTEGER,
    quantity INTEGER,
    unit_price REAL,
    discount REAL DEFAULT 0,
    FOREIGN KEY (order_id) REFERENCES orders(order_id),
    FOREIGN KEY (product_id) REFERENCES products(product_id)
);
""", conn)

print("Tables created successfully")
sql("SELECT name FROM sqlite_master WHERE type='table'", conn)

In [ ]:
# Populate with realistic data
np.random.seed(42)

# Customers
regions = ['North', 'South', 'East', 'West']
tiers = ['Gold', 'Silver', 'Bronze']
first_names = ['Alice', 'Bob', 'Carol', 'David', 'Eve', 'Frank', 'Grace', 'Hank',
               'Iris', 'Jack', 'Karen', 'Leo', 'Mona', 'Nick', 'Olivia', 'Paul',
               'Quinn', 'Rita', 'Sam', 'Tina', 'Uma', 'Vic', 'Wendy', 'Xander',
               'Yuki', 'Zara', 'Amit', 'Beth', 'Chris', 'Dana']

for i, name in enumerate(first_names, 1):
    signup = f"2023-{np.random.randint(1,13):02d}-{np.random.randint(1,29):02d}"
    conn.execute(
        "INSERT INTO customers VALUES (?, ?, ?, ?, ?, ?)",
        (i, name, f"{name.lower()}@email.com", np.random.choice(tiers),
         signup, np.random.choice(regions))
    )

# Products
products_data = [
    (1, 'Laptop Pro', 'Electronics', 1299.99, 780),
    (2, 'Wireless Mouse', 'Peripherals', 29.99, 10),
    (3, 'Mech Keyboard', 'Peripherals', 149.99, 55),
    (4, '4K Monitor', 'Electronics', 449.99, 220),
    (5, 'USB-C Hub', 'Accessories', 59.99, 18),
    (6, 'Webcam HD', 'Peripherals', 89.99, 30),
    (7, 'Standing Desk', 'Furniture', 399.99, 180),
    (8, 'Ergo Chair', 'Furniture', 299.99, 130),
    (9, 'Headset Pro', 'Accessories', 79.99, 25),
    (10, 'Laptop Stand', 'Accessories', 49.99, 15),
]
conn.executemany("INSERT INTO products VALUES (?, ?, ?, ?, ?)", products_data)

# Orders & Order Items
statuses = ['completed', 'completed', 'completed', 'completed', 'pending', 'cancelled', 'returned']
order_id = 1

for month in range(1, 13):
    n_orders = np.random.randint(15, 35)
    for _ in range(n_orders):
        cust_id = np.random.randint(1, 31)
        day = np.random.randint(1, 29)
        date = f"2024-{month:02d}-{day:02d}"
        status = np.random.choice(statuses)
        conn.execute("INSERT INTO orders VALUES (?, ?, ?, ?)",
                     (order_id, cust_id, date, status))
        
        # 1-3 items per order
        n_items = np.random.randint(1, 4)
        prods = np.random.choice(range(1, 11), n_items, replace=False)
        for pid in prods:
            qty = np.random.randint(1, 5)
            price = products_data[pid-1][3]
            discount = np.random.choice([0, 0, 0, 0.05, 0.10, 0.15])
            conn.execute("INSERT INTO order_items (order_id, product_id, quantity, unit_price, discount) VALUES (?, ?, ?, ?, ?)",
                         (order_id, int(pid), qty, price, discount))
        order_id += 1

conn.commit()

# Quick check
for table in ['customers', 'products', 'orders', 'order_items']:
    count = conn.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]
    print(f"{table}: {count} rows")

---
## 2. Basic Queries Refresher

In [ ]:
# SELECT with filtering and ordering
sql("""
SELECT name, tier, region, signup_date
FROM customers
WHERE tier = 'Gold'
ORDER BY signup_date
LIMIT 10
""", conn)

In [ ]:
# GROUP BY with aggregate functions
sql("""
SELECT 
    tier,
    COUNT(*) AS num_customers,
    region
FROM customers
GROUP BY tier, region
ORDER BY tier, num_customers DESC
""", conn)

In [ ]:
# HAVING — filter groups (not rows)
sql("""
SELECT 
    customer_id,
    COUNT(*) AS order_count
FROM orders
WHERE status = 'completed'
GROUP BY customer_id
HAVING order_count >= 10
ORDER BY order_count DESC
""", conn)

### WHERE vs HAVING
- **WHERE** filters individual rows *before* grouping
- **HAVING** filters groups *after* aggregation

---
## 3. JOINs

Combine data from multiple tables.

In [ ]:
# INNER JOIN — only matching rows from both tables
# "Show me all orders with customer details"
sql("""
SELECT 
    o.order_id,
    c.name AS customer_name,
    c.tier,
    o.order_date,
    o.status
FROM orders o
INNER JOIN customers c ON o.customer_id = c.customer_id
ORDER BY o.order_date DESC
LIMIT 10
""", conn)

In [ ]:
# LEFT JOIN — all rows from left table, matching from right
# "Show all customers, even those with no orders"
sql("""
SELECT 
    c.name,
    c.tier,
    COUNT(o.order_id) AS order_count
FROM customers c
LEFT JOIN orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.name, c.tier
ORDER BY order_count ASC
LIMIT 10
""", conn)

In [ ]:
# Multi-table JOIN — orders + customers + items + products
# "Revenue by customer and product"
sql("""
SELECT 
    c.name AS customer,
    p.name AS product,
    p.category,
    SUM(oi.quantity) AS total_qty,
    SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) AS revenue
FROM order_items oi
JOIN orders o ON oi.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
JOIN products p ON oi.product_id = p.product_id
WHERE o.status = 'completed'
GROUP BY c.name, p.name, p.category
ORDER BY revenue DESC
LIMIT 15
""", conn)

### JOIN Types Summary

| Type | Returns | Use When |
|------|---------|----------|
| INNER JOIN | Only matching rows | You need data from both tables |
| LEFT JOIN | All left + matching right | You want all records from the main table, even without matches |
| CROSS JOIN | Every combination (cartesian) | Generating combinations (rare in practice) |

*SQLite doesn't support RIGHT JOIN or FULL OUTER JOIN natively — use LEFT JOIN with swapped tables, or UNION.*

---
## 4. Subqueries

In [ ]:
# Subquery in WHERE — "Customers who spent above average"
sql("""
SELECT 
    c.name,
    c.tier,
    customer_totals.total_spent
FROM (
    SELECT 
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) AS total_spent
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY o.customer_id
) customer_totals
JOIN customers c ON customer_totals.customer_id = c.customer_id
WHERE customer_totals.total_spent > (
    SELECT AVG(sub.total_spent)
    FROM (
        SELECT SUM(oi2.quantity * oi2.unit_price * (1 - oi2.discount)) AS total_spent
        FROM order_items oi2
        JOIN orders o2 ON oi2.order_id = o2.order_id
        WHERE o2.status = 'completed'
        GROUP BY o2.customer_id
    ) sub
)
ORDER BY customer_totals.total_spent DESC
""", conn)

That query is hard to read! This is exactly why CTEs exist. ↓

---
## 5. CTEs (Common Table Expressions)

CTEs use `WITH` to define temporary named result sets. They make complex queries readable and reusable.

In [ ]:
# Same query as above, but readable with CTEs
sql("""
WITH customer_spending AS (
    SELECT 
        o.customer_id,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) AS total_spent
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY o.customer_id
),
avg_spending AS (
    SELECT AVG(total_spent) AS avg_spent
    FROM customer_spending
)
SELECT 
    c.name,
    c.tier,
    ROUND(cs.total_spent, 2) AS total_spent,
    ROUND(a.avg_spent, 2) AS avg_across_all
FROM customer_spending cs
JOIN customers c ON cs.customer_id = c.customer_id
CROSS JOIN avg_spending a
WHERE cs.total_spent > a.avg_spent
ORDER BY cs.total_spent DESC
""", conn)

In [ ]:
# Multiple CTEs chained: Monthly revenue with growth rate
sql("""
WITH monthly_revenue AS (
    SELECT 
        strftime('%Y-%m', o.order_date) AS month,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) AS revenue,
        SUM(oi.quantity * p.cost) AS total_cost
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    JOIN products p ON oi.product_id = p.product_id
    WHERE o.status = 'completed'
    GROUP BY strftime('%Y-%m', o.order_date)
),
with_profit AS (
    SELECT
        month,
        ROUND(revenue, 2) AS revenue,
        ROUND(revenue - total_cost, 2) AS profit,
        ROUND((revenue - total_cost) / revenue * 100, 1) AS margin_pct
    FROM monthly_revenue
)
SELECT * FROM with_profit
ORDER BY month
""", conn)

### CTE Tips
- Think of each CTE as a **building block** — solve one piece at a time
- CTEs can reference earlier CTEs in the same `WITH` block
- Much easier to debug than nested subqueries
- In production databases (PostgreSQL, etc.), CTEs can also be recursive

---
## 6. Window Functions

Window functions compute values across a set of rows **related to** the current row, without collapsing rows like GROUP BY.

Syntax: `function() OVER (PARTITION BY ... ORDER BY ...)`

In [ ]:
# ROW_NUMBER — assign a sequential number within each partition
# "Rank each customer's orders by date"
sql("""
SELECT 
    c.name,
    o.order_date,
    o.status,
    ROW_NUMBER() OVER (PARTITION BY c.customer_id ORDER BY o.order_date) AS order_num
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE c.name IN ('Alice', 'Bob')
ORDER BY c.name, order_num
""", conn)

In [ ]:
# RANK vs DENSE_RANK vs ROW_NUMBER
# "Rank products by total revenue"
sql("""
WITH product_revenue AS (
    SELECT 
        p.name,
        p.category,
        ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY p.product_id, p.name, p.category
)
SELECT 
    name,
    category,
    revenue,
    ROW_NUMBER() OVER (ORDER BY revenue DESC) AS row_num,
    RANK() OVER (ORDER BY revenue DESC) AS rank,
    DENSE_RANK() OVER (ORDER BY revenue DESC) AS dense_rank
FROM product_revenue
""", conn)

In [ ]:
# Running total — cumulative revenue over months
sql("""
WITH monthly AS (
    SELECT 
        strftime('%Y-%m', o.order_date) AS month,
        ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS revenue
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY strftime('%Y-%m', o.order_date)
)
SELECT 
    month,
    revenue,
    SUM(revenue) OVER (ORDER BY month) AS cumulative_revenue,
    ROUND(revenue * 100.0 / SUM(revenue) OVER (), 1) AS pct_of_total
FROM monthly
ORDER BY month
""", conn)

In [ ]:
# LAG / LEAD — access previous/next row values
# "Month-over-month revenue growth"
sql("""
WITH monthly AS (
    SELECT 
        strftime('%Y-%m', o.order_date) AS month,
        ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS revenue
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY strftime('%Y-%m', o.order_date)
)
SELECT 
    month,
    revenue,
    LAG(revenue) OVER (ORDER BY month) AS prev_month_revenue,
    ROUND(
        (revenue - LAG(revenue) OVER (ORDER BY month)) 
        / LAG(revenue) OVER (ORDER BY month) * 100, 
        1
    ) AS growth_pct
FROM monthly
ORDER BY month
""", conn)

In [ ]:
# PARTITION BY — window functions within groups
# "Top 2 products by revenue in each category"
sql("""
WITH product_revenue AS (
    SELECT 
        p.name AS product,
        p.category,
        ROUND(SUM(oi.quantity * oi.unit_price * (1 - oi.discount)), 2) AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY p.product_id, p.name, p.category
),
ranked AS (
    SELECT 
        product,
        category,
        revenue,
        ROW_NUMBER() OVER (PARTITION BY category ORDER BY revenue DESC) AS rank_in_category
    FROM product_revenue
)
SELECT * FROM ranked
WHERE rank_in_category <= 2
ORDER BY category, rank_in_category
""", conn)

### Window Functions Cheat Sheet

| Function | What It Does |
|----------|--------------|
| `ROW_NUMBER()` | Unique sequential number (1, 2, 3...) |
| `RANK()` | Rank with gaps on ties (1, 2, 2, 4) |
| `DENSE_RANK()` | Rank without gaps (1, 2, 2, 3) |
| `LAG(col, n)` | Value from n rows before |
| `LEAD(col, n)` | Value from n rows after |
| `SUM() OVER (ORDER BY ...)` | Running total |
| `AVG() OVER (PARTITION BY ...)` | Average within partition |
| `FIRST_VALUE()` / `LAST_VALUE()` | First/last value in window |

---
## 7. Practical Pattern: Full Business Report Query

Combining everything: CTEs + JOINs + window functions.

In [ ]:
# Customer Lifetime Value (CLV) report
sql("""
WITH order_values AS (
    -- Calculate value per order
    SELECT 
        o.order_id,
        o.customer_id,
        o.order_date,
        SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) AS order_value
    FROM order_items oi
    JOIN orders o ON oi.order_id = o.order_id
    WHERE o.status = 'completed'
    GROUP BY o.order_id, o.customer_id, o.order_date
),
customer_metrics AS (
    -- Aggregate per customer
    SELECT 
        customer_id,
        COUNT(*) AS total_orders,
        ROUND(SUM(order_value), 2) AS lifetime_value,
        ROUND(AVG(order_value), 2) AS avg_order_value,
        MIN(order_date) AS first_order,
        MAX(order_date) AS last_order,
        -- Days between first and last order
        JULIANDAY(MAX(order_date)) - JULIANDAY(MIN(order_date)) AS customer_lifespan_days
    FROM order_values
    GROUP BY customer_id
),
ranked_customers AS (
    -- Add ranking
    SELECT 
        cm.*,
        c.name,
        c.tier,
        c.region,
        DENSE_RANK() OVER (ORDER BY cm.lifetime_value DESC) AS value_rank,
        NTILE(4) OVER (ORDER BY cm.lifetime_value DESC) AS value_quartile
    FROM customer_metrics cm
    JOIN customers c ON cm.customer_id = c.customer_id
)
SELECT 
    name,
    tier,
    region,
    total_orders,
    lifetime_value,
    avg_order_value,
    value_rank,
    CASE value_quartile
        WHEN 1 THEN 'Top 25%'
        WHEN 2 THEN 'Upper Mid'
        WHEN 3 THEN 'Lower Mid'
        WHEN 4 THEN 'Bottom 25%'
    END AS value_segment
FROM ranked_customers
ORDER BY value_rank
LIMIT 15
""", conn)

---
## 8. SQL ↔ Pandas Bridge

Load SQL results into Pandas for further analysis and visualization.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Pull monthly revenue via SQL, visualize with Pandas/Matplotlib
monthly_df = sql("""
    WITH monthly AS (
        SELECT 
            strftime('%Y-%m', o.order_date) AS month,
            SUM(oi.quantity * oi.unit_price * (1 - oi.discount)) AS revenue,
            SUM(oi.quantity * p.cost) AS cost
        FROM order_items oi
        JOIN orders o ON oi.order_id = o.order_id
        JOIN products p ON oi.product_id = p.product_id
        WHERE o.status = 'completed'
        GROUP BY strftime('%Y-%m', o.order_date)
    )
    SELECT month, ROUND(revenue, 2) AS revenue, ROUND(revenue - cost, 2) AS profit
    FROM monthly
    ORDER BY month
""", conn)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_df['month'], monthly_df['revenue'], marker='o', label='Revenue')
ax.plot(monthly_df['month'], monthly_df['profit'], marker='s', label='Profit')
ax.set_title('Monthly Revenue & Profit')
ax.set_xlabel('Month')
ax.set_ylabel('Amount ($)')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

---
## 9. Practice Exercises

### Exercise 1: Order Frequency
Write a query that shows each customer's name, tier, total orders, and the average days between their orders.

In [ ]:
# Exercise 1: Your SQL here
# sql("""
# """, conn)

### Exercise 2: Category Market Share
Write a query using window functions that shows each product's revenue and its percentage of its category's total revenue.

In [ ]:
# Exercise 2: Your SQL here


### Exercise 3: Cohort Analysis
Using CTEs, find each customer's first order month (cohort), then show how many orders each cohort made in subsequent months.

In [ ]:
# Exercise 3: Your SQL here


In [ ]:
# Clean up
conn.close()

---
## Key Takeaways

1. **JOINs** combine tables. INNER for matches only, LEFT to keep all from the primary table.
2. **CTEs** (`WITH ... AS`) make complex queries readable. Build queries in layers.
3. **Window functions** compute across related rows without collapsing. Essential for rankings, running totals, and row comparisons.
4. **LAG/LEAD** are invaluable for period-over-period comparisons (growth rates, trends).
5. **SQL + Pandas** is a powerful combo: SQL for heavy aggregation, Pandas for further manipulation and visualization.